In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


In [48]:
#Ruta desde la raiz del proyecto
ruta_excel = Path("../data/raw/base_datos.xlsx")

#Leer el archivo
df = pd.read_excel(ruta_excel)

In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10757 entries, 0 to 10762
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   fecha_prestamo                 10757 non-null  datetime64[ns]
 1   capital_prestado               10757 non-null  float64       
 2   plazo_meses                    10757 non-null  int64         
 3   edad_cliente                   10757 non-null  int64         
 4   tipo_laboral                   10757 non-null  object        
 5   salario_cliente                10757 non-null  int64         
 6   total_otros_prestamos          10757 non-null  int64         
 7   puntaje_datacredito            10757 non-null  int64         
 8   cant_creditosvigentes          10757 non-null  int64         
 9   huella_consulta                10757 non-null  int64         
 10  saldo_mora                     10757 non-null  int64         
 11  saldo_total         

Revisamos la matriz de correlacion y se encontro que algunas columnas de datos tenian una correlacion muy alta, y lo mas recomendable es excluir estas variables ya que presentarian problemas de alta multicolinealidad, (Redundancia de informacion) o posible fuga de datos 

Se eliminaran las variables:

puntaje
creditos_sectorFinanciero
cuota_pactada
saldo_principal
saldo_mora_codeudor

In [72]:
df = df.drop(columns=['puntaje', 'creditos_sectorFinanciero', 'cuota_pactada', 'saldo_principal', 'saldo_mora_codeudor'])

Vamos a realizar toda la gestion de los nulos en la base de datos 

In [83]:
print(df.isnull().sum())

fecha_prestamo                   0
capital_prestado                 0
plazo_meses                      0
edad_cliente                     0
tipo_laboral                     0
salario_cliente                  0
total_otros_prestamos            0
puntaje_datacredito              0
cant_creditosvigentes            0
huella_consulta                  0
saldo_mora                       0
saldo_total                      0
creditos_sectorCooperativo       0
creditos_sectorReal              0
promedio_ingresos_datacredito    0
tendencia_ingresos               0
Pago_atiempo                     0
tipo_credito_4                   0
tipo_credito_9                   0
tipo_credito_10                  0
tipo_credito_Otros               0
dtype: int64


PUNTAJE DATACREDITO

revisando los datos que tiene mi columna de puntaje datacredito, encontramos que algunos de estos son de tipo float los cuales no tienen ningun valor decimal real, ent se decidio convertir a tipo entero toda la columna, pero para poder hacer esto, se encontro que habian 6 valores nulos, los cuales NO podemos asignarles un valor como 0, ya que tambien esta el puntaje 0, ent asi se decidio eliminar a estos 6 registros, ya que es un porcentaje menor al 0.06% de los datos

In [65]:
df['puntaje_datacredito'].value_counts()

puntaje_datacredito
0.0      145
802.0    102
783.0    102
791.0    100
804.0     98
        ... 
618.0      1
404.0      1
640.0      1
555.0      1
923.0      1
Name: count, Length: 315, dtype: int64

In [59]:
# 1. Aislamos la columna eliminando temporalmente los nulos
serie_sin_nulos = df['puntaje_datacredito'].dropna()

# 2. Aplicamos la evaluación de decimales sobre esa misma serie
con_decimales = serie_sin_nulos[serie_sin_nulos % 1 != 0]

print("Registros con decimales reales:", len(con_decimales))

Registros con decimales reales: 0


In [ ]:
# 1. Eliminar las 6 filas con nulos en esta columna específica
df = df.dropna(subset=['puntaje_datacredito'])

# 2. Convertir los datos restantes a números enteros
df['puntaje_datacredito'] = df['puntaje_datacredito'].astype(int)

# 3. Verificar que la limpieza fue exitosa
print("Nulos restantes:", df['puntaje_datacredito'].isnull().sum())
print("Tipo de dato:", df['puntaje_datacredito'].dtype)

Nulos restantes: 0
Tipo de dato: int64


SALDO MORA

Para la columna saldo_mora se identifico que mas de 10.000 de los datos contenian el valor 0, y los demas eran numeros dispersos, frente a esto se tomo la decision de convertir la variable saldo_mora a tipo binario, 156 valores nulos los asignamos como que no tenian saldo en mora, ya que no afectaban en la generacion del modelo y ademas se podia interpretar como la ausencia al igual que el 0 de saldo en mora, despues los demas valores que eran diferentes y mayores a 0, lo que significaba que si habia saldo en mora, les asignamos un valor de 1, a pesar de que solo eran 55 registros los cuales tenian un saldo en mora se evidencio que el tener un saldo en mora aumentaba drasticamente si si pagaban a tiempo, pasando de 4.58% de imcumplir su pago teniendo el saldo en mora en 0, a 36.36% de incumplir su pago teniendo su saldo en mora en 1 

In [60]:
# 1. Rellenar los 156 valores nulos con 0
df['saldo_mora'] = df['saldo_mora'].fillna(0)

# 2. Convertir a formato binario (1 si tiene saldo mayor a 0, 0 si no)
df['saldo_mora'] = (df['saldo_mora'] > 0).astype(int)

In [61]:
# Reemplaza 'tu_variable_objetivo' por el nombre real de la columna a predecir
print(pd.crosstab(df['saldo_mora'], df['Pago_atiempo'], normalize='index'))

Pago_atiempo         0         1
saldo_mora                      
0             0.045854  0.954146
1             0.363636  0.636364


SALDO TOTAL

Los 150 registros nulos de la comuna saldo total se rellenaron con la media (promedio de los datos) para asi no generar una desviacion o sesgo en el modelo

In [70]:
df['saldo_total'].value_counts()

saldo_total
0.0        789
39.0        10
35.0        10
29.0         9
53.0         7
          ... 
2419.0       1
1374.0       1
89898.0      1
59.0         1
18035.0      1
Name: count, Length: 8858, dtype: int64

In [ ]:
# 1. Calcular la media (Pandas ignora automáticamente los 150 nulos al calcularla)
media_columna = df['saldo_total'].mean()

# 2. Imputar los valores nulos con la media
df['saldo_total'] = df['saldo_total'].fillna(media_columna)

Valores nulos restantes: 0


PROMEDIO INGRESOS DATACREDITO

Con los 2.925 datos nulos de la columna, si los eliminabamos generaba un gran porcentaje de perdida de datos, ent revisando se opto por inputar los valores nulos con la media (promedio de los datos) de los datos 

In [78]:
# 1. Calcular la media (Pandas ignora automáticamente los 150 nulos al calcularla)
media_columna = df['promedio_ingresos_datacredito'].mean()

# 2. Imputar los valores nulos con la media
df['promedio_ingresos_datacredito'] = df['promedio_ingresos_datacredito'].fillna(media_columna)

TENDENCIA INGRESOS 

Se evidencio que en los datos de la columna habian datos atipicos com numericos, 57 eran numericos, y el resto que eran al rededor de 2.900 a 3.000, y esos nulos se llenaron con la moda

In [82]:
# 1. Definir las categorías válidas
categorias_validas = ['Creciente', 'Decreciente', 'Estable']

# 2. Convertir a nulo (NaN) cualquier dato que no pertenezca a las categorías válidas
df.loc[~df['tendencia_ingresos'].isin(categorias_validas), 'tendencia_ingresos'] = np.nan

# 3. Calcular la moda (pandas ignora los NaN automáticamente)
moda_tendencia = df['tendencia_ingresos'].mode()[0]

# 4. Imputar los valores nulos con la moda
df['tendencia_ingresos'] = df['tendencia_ingresos'].fillna(moda_tendencia)

# 5. Verificar que solo queden las 3 categorías y los datos se hayan sumado a la moda
print(df['tendencia_ingresos'].value_counts())

tendencia_ingresos
Creciente      8278
Decreciente    1291
Estable        1188
Name: count, dtype: int64


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10757 entries, 0 to 10762
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   fecha_prestamo                 10757 non-null  datetime64[ns]
 1   capital_prestado               10757 non-null  float64       
 2   plazo_meses                    10757 non-null  int64         
 3   edad_cliente                   10757 non-null  int64         
 4   tipo_laboral                   10757 non-null  object        
 5   salario_cliente                10757 non-null  int64         
 6   total_otros_prestamos          10757 non-null  int64         
 7   puntaje_datacredito            10757 non-null  int64         
 8   cant_creditosvigentes          10757 non-null  int64         
 9   huella_consulta                10757 non-null  int64         
 10  saldo_mora                     10757 non-null  int64         
 11  saldo_total         

In [63]:
# 1. Identificar las categorías con baja frecuencia (menos de 50 registros)
conteos = df['tipo_credito'].value_counts()
categorias_minoritarias = conteos[conteos < 50].index

# 2. Reemplazar esos valores atípicos por la categoría 'Otros'
df['tipo_credito'] = df['tipo_credito'].replace(categorias_minoritarias, 'Otros')

df = pd.get_dummies(df, columns=['tipo_credito'], dtype=int)

In [64]:
df.head()

,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,...,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo,tipo_credito_4,tipo_credito_9,tipo_credito_10,tipo_credito_Otros
0,2025-01-07 14:40:36,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,795.0,...,0,0,0,916148.0,Creciente,1,1,0,0,0
1,2025-01-09 11:18:48,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,796.0,...,0,0,0,4473774.0,Creciente,1,1,0,0,0
2,2024-12-26 18:52:57,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,727.0,...,2,0,1,NaN,NaN,1,0,1,0,0
3,2024-12-04 14:20:50,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,864.0,...,1,0,6,NaN,NaN,1,0,1,0,0
4,2025-04-30 18:41:57,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,771.0,...,0,0,0,61000.0,Creciente,1,1,0,0,0
